In [1]:

import pandas as pd
from scipy.stats import skew, kurtosis
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score


In [2]:

# Load datasets
uci = load_diabetes(as_frame=True).frame

pima = pd.read_csv(
    "https://raw.githubusercontent.com/anshulsavla/Pima-Indian-Diabetes-Dataset/master/diabetes.csv"
)


In [3]:

def univariate(df, name):
    print("\n", name)

    s = df.select_dtypes("number")

    print(pd.DataFrame({
        "Frequency": s.nunique(),
        "Mean": s.mean(),
        "Median": s.median(),
        "Mode": s.mode().iloc[0],
        "Variance": s.var(),
        "Std Dev": s.std(),
        "Skewness": s.apply(skew),
        "Kurtosis": s.apply(kurtosis)
    }).round(2).to_string())


univariate(uci, "UCI Diabetes")
univariate(pima, "Pima Diabetes")



 UCI Diabetes
        Frequency    Mean  Median   Mode  Variance  Std Dev  Skewness  Kurtosis
age            58   -0.00    0.01   0.02      0.00     0.05     -0.23     -0.68
sex             2    0.00   -0.04  -0.04      0.00     0.05      0.13     -1.98
bmi           163   -0.00   -0.01  -0.03      0.00     0.05      0.60      0.08
bp            100   -0.00   -0.01  -0.04      0.00     0.05      0.29     -0.54
s1            141   -0.00   -0.00  -0.04      0.00     0.05      0.38      0.22
s2            302    0.00   -0.00  -0.00      0.00     0.05      0.44      0.58
s3             63   -0.00   -0.01  -0.01      0.00     0.05      0.80      0.96
s4             66   -0.00   -0.00  -0.04      0.00     0.05      0.73      0.43
s5            184    0.00   -0.00  -0.02      0.00     0.05      0.29     -0.15
s6             56    0.00   -0.00   0.00      0.00     0.05      0.21      0.22
target        214  152.13  140.50  72.00   5943.33    77.09      0.44     -0.89

 Pima Diabetes
         

In [4]:

def linear(df, x, y):

    X_train, X_test, y_train, y_test = train_test_split(
        df[[x]], df[y],
        test_size=0.2,
        random_state=42
    )

    m = LinearRegression().fit(X_train, y_train)

    pred = m.predict(X_test)

    return (
        round(r2_score(y_test, pred), 3),
        round(mean_squared_error(y_test, pred) ** 0.5, 3)
    )


print("\nBivariate Linear Regression (R2, RMSE)")

print("UCI BMI -> Target:",
      linear(uci, "bmi", "target"))

print("Pima BMI -> Glucose:",
      linear(pima, "BMI", "Glucose"))



Bivariate Linear Regression (R2, RMSE)
UCI BMI -> Target: (0.233, 63.732)
Pima BMI -> Glucose: (0.076, 30.499)


In [5]:

def logistic(X, y):

    a, b, c, d = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    m = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    m.fit(a, c)

    return round(accuracy_score(d, m.predict(b)), 3)


print("\nBivariate Logistic Regression Accuracy")

print("UCI BMI -> High/Low Target:",
      logistic(
          uci[["bmi"]],
          (uci.target > uci.target.median()).astype(int)
      ))

print("Pima Glucose -> Outcome:",
      logistic(pima[["Glucose"]], pima["Outcome"]))



Bivariate Logistic Regression Accuracy
UCI BMI -> High/Low Target: 0.652
Pima Glucose -> Outcome: 0.708


In [6]:

a, b, c, d = train_test_split(
    uci.drop(columns="target"),
    uci.target,
    test_size=0.2,
    random_state=42
)

m = LinearRegression().fit(a, c)

print("\nUCI Multiple Linear Regression (R2, RMSE):",
      round(r2_score(d, m.predict(b)), 3),
      round(mean_squared_error(d, m.predict(b)) ** 0.5, 3))



UCI Multiple Linear Regression (R2, RMSE): 0.453 53.853


In [7]:

features = pima.drop(columns="Outcome")

a, b, c, d = train_test_split(
    features,
    pima.Outcome,
    test_size=0.2,
    random_state=42,
    stratify=pima.Outcome
)

m = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

m.fit(a, c)

print("Pima Multiple Logistic Regression Accuracy:",
      round(accuracy_score(d, m.predict(b)), 3))


Pima Multiple Logistic Regression Accuracy: 0.714


In [8]:

print("\nDataset Comparison")

print("UCI shape:", uci.shape, "| Pima shape:", pima.shape)

print("UCI target: continuous diabetes progression")

print("Pima target: binary diabetes outcome (0/1)")



Dataset Comparison
UCI shape: (442, 11) | Pima shape: (768, 9)
UCI target: continuous diabetes progression
Pima target: binary diabetes outcome (0/1)
